# APMS schema-aware forecasting playground

This notebook exercises the advisory sequence model on controlled synthetic examples. Synthetic scores are software-test data only; they do not establish institutional validity or expected accuracy. For a reproducible local run, open this notebook with the repository `.venv` kernel (Windows: `C:\apms\APMS\.venv\Scripts\python.exe`). Dependencies are provisioned by the repository AI service setup; do not install globally. Every execution creates a unique immutable bundle under `services/ai/artifacts/notebook-<uuid>`.

Real-data work must use approved de-identified records, explicit stable group keys and a held-out guard group set. Never mark generated examples as real. For online candidate updates, use the CLI's `online-update` protocol only with genuinely real candidate and held-out records; do not manufacture `origin="real"` labels.

## A short guide to the learning steps

`generate(count=80)` creates 80 synthetic example windows; it is a data volume, not a score or accuracy percentage. The model fits weights using the train partition. Validation helps compare settings and stop training. Calibration is held aside to adjust interval/risk outputs after choosing a model. The test partition is for final evaluation and should not guide repeated model changes. Splitting by group keeps the same learner/group out of multiple partitions.

Training loss is an optimization objective, not an accuracy percentage. Quantile/pinball loss trains interval forecasts; the optional numeric center loss adds a penalty for the median prediction. With `numeric_center_loss_kind="mse"`, squared normalized errors emphasize large misses, and that mixed-loss center is not a pure conditional median. RMSE is a separate evaluation measure: the square root of average squared error in the target's raw units. Check counts and baselines alongside every metric.

The default demo below stays unchanged. These are examples of opt-in experiment settings (set them explicitly in a separate run/profile after choosing a controlled comparison):

```python
# Optional feature/optimization/selection experiments; not enabled in this demo.
# numeric_history_context=True
# numeric_center_loss_kind="mse"
# selection_metric="numeric_rmse"
```

These switches change inputs, the loss objective, or checkpoint selection. Compare them on the same train/validation groups, keep calibration/test untouched until the choice is frozen, and do not treat the option name as evidence of a gain.

For a plain-language explanation of metrics and the current evidence, see [How the APMS forecasting model learns](../../../docs/AI_MODEL_TRAINING_EXPLAINED.md). For safe artifact handling, see [AI workspace files and cleanup](../../../docs/AI_MODEL_FILES_AND_CLEANUP.md).

In [ ]:
from pathlib import Path
from dataclasses import replace
import json, sys, uuid

ROOT = Path.cwd().resolve()
while not (ROOT / "services" / "ai" / "src" / "apms_ai").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "services" / "ai" / "src" / "apms_ai").is_dir():
    raise RuntimeError("Run this notebook from the APMS repository or a subdirectory")
sys.path.insert(0, str(ROOT / "services" / "ai" / "src"))
ARTIFACTS = ROOT / "services" / "ai" / "artifacts" / f"notebook-{uuid.uuid4().hex}"
ARTIFACTS.mkdir(parents=True, exist_ok=False)
SEED = 42
RUN_OPTIONAL = False  # Enable the separate optional workflow section explicitly.
print({"root": str(ROOT), "artifact_directory": str(ARTIFACTS), "seed": SEED})

In [ ]:
from apms_ai.synthetic import generate
from apms_ai.training import train
from apms_ai.evaluation import score_bundle
from apms_ai.calibration import calibrate_bundle
from apms_ai.lifecycle import load_bundle
from apms_ai.inference import predict_example
from apms_ai.diagnostics import diagnose_bundle

examples = generate(count=80, seed=SEED)
dataset_path = ARTIFACTS / "controlled-synthetic.jsonl"
dataset_path.write_text("\n".join(json.dumps({"history": ex.history, "queries": ex.queries,
    "fields": [field.__dict__ for field in ex.fields], "labels": ex.labels, "group": ex.group,
    "origin": ex.origin, "schema": ex.schema}, allow_nan=False) for ex in examples) + "\n", encoding="utf-8")
print(f"Loaded {len(examples)} examples; origin={examples[0].origin!r}; schema keys={sorted(examples[0].schema)}")
print("Declared fields:", [(field.name, field.kind, len(field.categories)) for field in examples[0].fields])
print("Grading snapshot keys:", sorted(examples[0].schema.get("gradingSystem", {})))
print("Evaluation snapshot keys:", sorted(examples[0].schema.get("evaluationCriteria", {})))

In [ ]:
bundle = ARTIFACTS / "trained-model"
training_result = train(examples, bundle, seed=SEED, epochs=2, batch_size=8, width=32)
training_result

In [ ]:
calibrated_bundle = ARTIFACTS / "calibrated-model"
calibration_report = calibrate_bundle(bundle, examples, coverage=0.8, output=calibrated_bundle)
evaluation = score_bundle(calibrated_bundle, examples, role="test")
print("Per-query metrics:")
evaluation["by_query"]

In [ ]:
state, metadata = load_bundle(calibrated_bundle)
request_example = replace(examples[-1], labels=[])
typed_forecast = predict_example(state, metadata, request_example)
typed_forecast

In [ ]:
diagnostics = diagnose_bundle(calibrated_bundle, examples, backward_check=False)
diagnostics

## Optional lifecycle playground

Set `RUN_OPTIONAL = True` above to run masked pretraining, two-trial tuning, fine-tuning and teacher/student distillation, then call SHAP and dynamic ONNX export through the public CLI. INT8 export has a held-out accuracy/parity gate and may correctly refuse a candidate. These optional steps take longer and create additional immutable bundles under the same unique artifact directory.

In [ ]:
if RUN_OPTIONAL:
    import subprocess
    from apms_ai.pretraining import pretrain
    from apms_ai.tuning import tune
    from apms_ai.training import train as fit
    python = sys.executable
    pretrain_result = pretrain(examples, ARTIFACTS / "pretrained", seed=SEED, epochs=1, width=32)
    tuning_result = tune(dataset_path, ARTIFACTS / "tuning", seed=SEED, trials=2, epochs=1)
    finetune_result = fit(examples, ARTIFACTS / "finetuned", seed=SEED, epochs=1, width=32,
                          init_from=pretrain_result["bundle"])
    student_result = fit(examples, ARTIFACTS / "distilled-student", seed=SEED, epochs=1, width=32,
                         teacher_bundle=tuning_result["bundle"], distill_weight=0.5)
    commands = [
        [python, "-m", "apms_ai.cli", "explain", "--bundle", str(calibrated_bundle), "--data", str(dataset_path), "--output", str(ARTIFACTS / "explanation.json"), "--max-evaluations", "64"],
        [python, "-m", "apms_ai.cli", "export-onnx", "--bundle", str(calibrated_bundle), "--data", str(dataset_path), "--output", str(ARTIFACTS / "model.onnx")],
    ]
    for command in commands:
        subprocess.run(command, check=True, cwd=ROOT)
    optional_results = {"pretrain": pretrain_result, "tuning": tuning_result,
                        "finetune": finetune_result, "distillation": student_result}
    optional_results

## Interpreting the output

The evaluation section is restricted to the saved test split and reports metrics by output field, horizon and legal domain. Calibration is fitted only on the saved calibration partition and can remain unavailable when finite-sample evidence is too small. The typed inference example removes future labels before prediction. Diagnostics show data/model checks without exposing learner identifiers. All outputs are advisory; an educator remains responsible for assessment and intervention.

For approved real data, ingest JSON/JSONL with declared field contracts and group keys or use `apms-ai ingest-csv` for the supported flat format. Keep group-disjoint train, validation, calibration and test partitions. Review the provider privacy contract before inference. Candidate updates require genuinely real new observations plus a genuinely real, disjoint external guard; the tool retains a candidate bundle and does not promote it automatically.